In [ ]:
# ===================== Imports =====================
import argparse, sys, os, glob, json, math, datetime, torch, yaml
import matplotlib.pyplot as plt, numpy as np, pandas as pd
from scipy.spatial.distance import pdist


from collections import defaultdict
from scipy.stats import norm, pearsonr

from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LinearRegression
from sklearn.utils import resample
from pathlib import Path

sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/src/model/')
sys.path.append("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/")

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

from encoders import *
from utls.plotting import ensure_dir
from utls.loading import load_results_with_exclusion_2, move_sequences_to_used, load_results_with_exclusion_no_dropping
from utls.runners import run_experiment_scores
from utls.runners_v2 import (
    run_experiment_grid,
    run_experiment_scores_v2,
    run_experiment_scores_itemwise_v2,
    run_experiment_itemwise_hits_fas,
    make_noise_schedule
)

from utls.analysis_helpers import rocs_across_noise, convert_human_to_model_struct, compute_scaling_vs_human, convert_human_to_model_struct
from utls.analysis_helpers import auroc_to_dprime, compute_model_dprime_curve
from utls.analysis_helpers import roc_for_isi, auroc_to_dprime
from utls.plotting import plot_across_noise, plot_noise_overlays
from utls.io_utils import make_model_save_dir, save_all_figures, save_single_figure, save_runs_summary
from utls.roc_utils import roc_from_arrays 
from utls.runners_utils import *

from datetime import date

today = date.today()
current_date_string = today.isoformat()

def load_config(cfg_path=None):
    """
    Load YAML config.
    Priority:
      1. cfg_path argument (if provided)
      2. sys.argv[1]
    """
    if cfg_path is None:
        if len(sys.argv) != 2:
            raise RuntimeError("Usage: python main.py path/to/run.yaml")
        cfg_path = sys.argv[1]

    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)

    with open(cfg_path, "r") as f:
        return yaml.safe_load(f), cfg_path

def median_pairwise_distance(X, metric="euclidean", n_samples=500, seed=0):
    """
    Estimate the median pairwise distance under a given metric.

    Args:
        X (array-like): shape (N, D). Must already be preprocessed
                        appropriately for the metric (e.g., L2-normalized for cosine).
        metric (str): distance metric for scipy.spatial.distance.pdist
                      (e.g., 'euclidean', 'cosine', 'mahalanobis').
        n_samples (int): number of items to subsample for efficiency.
        seed (int): RNG seed.

    Returns:
        float: median pairwise distance.
    """
    rng = np.random.default_rng(seed)
    N = X.shape[0]

    idx = rng.choice(N, size=min(n_samples, N), replace=False)
    Xs = X[idx]

    dists = pdist(Xs, metric=metric)
    d50 = np.median(dists)

    return float(d50)


tasks = {
    0: "ind-nature",
    1: "global-music",
    2: "atexts",
}

print(current_date_string)

In [ ]:
model_cfg, model_cfg_path = load_config("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/model_yamls/all_nn_v4/run_000001.yaml")

noise_cfg = model_cfg["noise_model"]
print(model_cfg)

assert "t_step" in noise_cfg, "t_step is needed for two-regime"

In [ ]:
model_cfg['noise_model']['sigma0_max'] = 3.0
model_cfg['noise_model']['sigma1_max'] = 0.5
model_cfg['noise_model']['t_step'] = 7

print(model_cfg['noise_model'])

In [ ]:
# ---------------------------
# SETTINGS (from YAML)
# ---------------------------
# ---- experiment ----
exp_cfg = model_cfg["experiment"]

which_task = exp_cfg["which_task"]
is_multi   = exp_cfg["is_multi"]
which_isi  = exp_cfg.get("which_isi", None)

if is_multi:
    isis = [0, 1, 2, 4, 8, 16, 32, 64]
else:
    assert which_isi is not None, "which_isi required if not multi-ISI"
    isis = [0, which_isi]

# ---- metric ----
metric = model_cfg["metric"]

# ---- noise model ----
noise_cfg = model_cfg["noise_model"]
noise_mode = noise_cfg["name"]
print(noise_cfg)

# ---- representation ----
repr_cfg = model_cfg["representation"]

encoder_type = repr_cfg["type"]
layer   = repr_cfg.get("layer", None)
PC_DIMS = repr_cfg.get("pc_dims", None)

# ---------------------------
# 1. Load data
# ---------------------------
exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = load_experiment_data(
    which_task, which_isi, is_multi, old=False)

# ---------------------------
# 2. Human curve
# ---------------------------
human_curve = compute_human_curve(human_runs, is_multi, which_isi)
results_root = model_cfg["results_root"]
tag = model_cfg.get("tag", "untagged")

if noise_mode == "two-regime":
    assert "t_step" in noise_cfg, "two-regime requires t_step"
    t_step = noise_cfg["t_step"]
    noise_tag = f"{noise_mode}_t{t_step}"
else:
    noise_tag = noise_mode

save_figs = (
    f"{results_root}/"
    f"figures/test_v2/"
    f"{task_name}/"
    f"{encoder_type}/"
    f"{metric}/"
    f"{noise_tag}/"
)

save_fits = f"{results_root}/fits/test_v2"


print(save_figs, save_fits)
ensure_dir(save_figs)

ensure_dir(save_fits)


In [ ]:
save_figs = (
    f"{results_root}/"
    f"figures"
    f"norm_test_{current_date_string}/"
    f"{task_name}/"
    f"{encoder_type}/"
    f"{metric}/"
    f"{noise_tag}/"
)

save_fits = f"{results_root}/fits/norm_test_{current_date_string}"
save_fits_all = f"{results_root}/fits/norm_test_{current_date_string}"

ensure_dir(save_figs)
ensure_dir(save_fits_all)
ensure_dir(save_fits)


encoder_type = repr_cfg["type"]     # e.g. "resnet50"
layer        = repr_cfg.get("layer")
pc_dims      = repr_cfg.get("pc_dims")

print(layer)

NN_ENCODERS = {"kell2018", "resnet50"}

encoder_task = (
    "word_speaker_audioset"
    if encoder_type in NN_ENCODERS
    else "audioset"
)

encoder_cfg = dict(
    encoder_type=encoder_type,      # e.g. "resnet50"
    model_name=encoder_type,        # same by design
    task=encoder_task,
    statistics_dict=statistics_set.statistics,
    model_params=model_params,
    pc_dims=pc_dims,
    sr=20000,
    duration=2.0,
    rms_level=0.05,
    device="cuda",
)

# ---- representation-specific fields ----
if encoder_type in ("kell2018", "resnet50"):
    encoder_cfg["layer"] = layer
    assert layer is not None, f"layer required for {encoder_type}"

if encoder_type == "texture":
    encoder_cfg["pc_dims"] = pc_dims
    assert pc_dims is not None, "pc_dims required for texture encoder"

encoder_name = make_encoder_name(encoder_cfg)
print("Encoder name:", encoder_name)

encoder = build_encoder(encoder_cfg)
X = encode_stimuli(encoder, all_files)

# optional
X = X / (torch.sqrt(torch.mean(X**2, dim=0)) + 1e-12)

# mandatory
X = X / (torch.linalg.norm(X, dim=1, keepdim=True) + 1e-12)

noise_cfg = model_cfg["noise_model"]
noise_mode = noise_cfg["name"]

# ---------------------------
# Metric-specific distance scale
# ---------------------------

X_np = X.detach().cpu().numpy()  # pdist expects numpy
d50 = median_pairwise_distance(X_np, metric=metric)

print(f"[d50] median {metric} distance = {d50:.4f}")


# ---------------------------
# Noise parameter bounds (from YAML)
# ---------------------------

param_bounds = {
    "sigma0": (noise_cfg["sigma0_min"] * d50, noise_cfg["sigma0_max"] * d50),
}

if noise_mode == "two-regime":
    param_bounds["sigma1"] = (
        noise_cfg["sigma1_min"] * d50,
        noise_cfg["sigma1_max"] * d50,
    )
    param_bounds["t_step"] = (
        noise_cfg["t_step"],
        noise_cfg["t_step"],
    )

import time
start_time = time.time()
opt_results = random_search_gridlike_v2(
    n_samples=20,
    param_bounds=param_bounds,
    noise_mode=noise_mode,
    metric=metric,
    X0=X,
    name_to_idx=name_to_idx,
    experiment_list=exp_list[:model_cfg['experiment']['n_seqs']],
    isis=isis,
    human_curve=human_curve,
    layer=encoder_name,          # legacy arg, keep for now
    encoder_name=encoder_name,   # canonical identifier
    hr_task_name=hr_task_name,
    debug=False,
)
print("--- %s seconds ---" % (time.time() - start_time))


# best_fits = generate_and_plot_model_decay_summary_v2(
#     opt_results,
#     human_curve,
#     isis,
#     savedir=save_figs,
#     max_rows=1,
#     verbose=True, hr_task_name=hr_task_name, encoder_name=encoder_name)

# summary_list = save_best_models(best_fits, save_fits_all, prefix=f"{task_name}-{encoder_name}")

# plot_best_model_histograms(best_fits, isis, save_figs)


In [ ]:
def refine_search_local(
    base_results,
    n_refine,
    param_bounds,
    *,
    noise_mode,
    top_k=10,
    jitter_frac=0.15,
    rng=None
):
    rng = np.random.default_rng(rng)

    # Sort by coarse fit (lower is better)
    base_results = sorted(base_results, key=lambda r: r["nmse"])

    # Keep only top-K elites
    elites = base_results[:top_k]

    refined_params = []

    for base in elites:
        p0 = base["params"]
        print("p0", p0)

        for _ in range(n_refine):
            p = {}
            for k, (lo, hi) in param_bounds.items():
                span = hi - lo
                val = p0[k] + rng.normal(scale=jitter_frac * span)
                p[k] = np.clip(val, lo, hi)

            if noise_mode == "two-regime":
                if p["sigma0"] <= p["sigma1"]:
                    continue

            refined_params.append(p)

    return refined_params

In [ ]:
refined_params = refine_search_local(
    base_results=opt_results,
    top_k=3,
    n_refine=20,
    param_bounds=param_bounds,
    noise_mode=noise_mode,
)



In [ ]:
#list(refined_params)

In [ ]:
def evaluate_params(
    params_list,
    *,
    noise_mode,
    metric,
    X0,
    name_to_idx,
    experiment_list,
    isis,
    human_curve,
    layer,
    encoder_name,
    hr_task_name,
    debug=False
):
    results = []

    for params in params_list:
        params = dict(params)  # defensive copy

        params.update({
            "noise_mode": noise_mode,
            "metric": metric,
            "layer": layer,
            "encoder": encoder_name,
            "stimulus_set": hr_task_name,
        })

        run_out = run_experiment_scores(
            sigma0=params["sigma0"],
            sigma1=params.get("sigma1", None),
            t_step=params.get("t_step", None),
            rate=params.get("rate", None),
            noise_mode=noise_mode,
            metric=metric,
            X0=X0,
            name_to_idx=name_to_idx,
            experiment_list=experiment_list,
            debug=debug,
        )


        model_dp = compute_model_dprime_for_run(run_out, isis)

        results.append({
            "params": params,
            "results": run_out,
            "model_dp": model_dp,
            "nmse": compute_nmse(model_dp, human_curve),
            "nmse_no_0": compute_nmse(model_dp, human_curve, start_idx=1),
            "mse": compute_mse(model_dp, human_curve),
            "mse_no_0": compute_mse(model_dp, human_curve, start_idx=1),
        })

    return results

In [ ]:
# 3) Run full experiments on refined params
fine_results = evaluate_params(
    refined_params,
    noise_mode=noise_mode,
    metric=metric,
    X0=X,
    name_to_idx=name_to_idx,
    experiment_list=exp_list[16:25],
    isis=isis,
    human_curve=human_curve,
    layer=layer,
    encoder_name=encoder_name,
    hr_task_name=hr_task_name,
)

summary_list = save_best_models(best_fits, save_fits, prefix=f"{task_name}-{encoder_name}")


In [ ]:
best_fits = generate_and_plot_model_decay_summary_v2(
    fine_results,
    human_curve,
    isis,
    savedir=save_figs,
    max_rows=1,
    verbose=True, hr_task_name=hr_task_name, encoder_name=encoder_name)

summary_list = save_best_models(best_fits, save_fits_all, prefix=f"{task_name}-{encoder_name}")

plot_best_model_histograms(best_fits, isis, save_figs)